In [4]:
import torch
import torch.nn as nn 
import tiktoken 

In [ ]:
class SingleHeadAttention(nn.Module): 
  def __init__(self, d_in, d_out, context_length, drop_rate, qkv_bias=False): 
    super().__init__()
    self.d_in = d_in
    self.d_out = d_out
    self.context_length = context_length
    self.drop_rate = drop_rate

    # weight matrices: (8, 16), PyTorch converts to (16, 8) by default
    self.W_Q = nn.Linear(d_in, d_out, bias=qkv_bias)                    
    self.W_K = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_V = nn.Linear(d_in, d_out, bias=qkv_bias)

    # mask fill 
    self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1).bool())

    # dropout 
    self.dropout = nn.Dropout(drop_rate)

    # print("W_Q:", self.W_Q.weight.shape)
    # print("W_K:", self.W_K.weight.shape)
    # print("W_V:", self.W_V.weight.shape)
  
  def forward(self, x):
    batch, num_tokens, d_in = x.shape 

    queries = self.W_Q(x)
    keys = self.W_K(x)
    values = self.W_V(x)

    print("queries:", queries.shape)
    print("keys:", keys.shape)
    print("values:", values.shape)
    
    # attention score 
    attn_score = queries @ keys.transpose(-2, -1)
    scaled_score = attn_score / (self.d_out ** 0.5)

    # applying causal attention with attention mask 
    scaled_score = scaled_score.masked_fill(self.mask[:num_tokens, :num_tokens], float("-inf"))

    attn_weight = torch.softmax(scaled_score, dim=-1)
    attn_weight = self.dropout(attn_weight)

    # return context vector
    return attn_weight @ values 

In [ ]:
torch.manual_seed(420)
x = torch.randn(2, 4, 8)          # b=2, num_tokens=4, d_in=8
attn = SingleHeadAttention(d_in=8, d_out=16, context_length=1024, drop_rate=0.1)
out = attn(x)
out.shape 


W_Q: torch.Size([16, 8])
W_K: torch.Size([16, 8])
W_V: torch.Size([16, 8])
queries: torch.Size([2, 4, 16])
keys: torch.Size([2, 4, 16])
values: torch.Size([2, 4, 16])
tensor([[[1.0000, 0.0000, 0.0000, 0.0000],
         [0.2942, 0.7058, 0.0000, 0.0000],
         [0.4174, 0.2390, 0.3436, 0.0000],
         [0.2336, 0.2936, 0.2641, 0.2086]],

        [[1.0000, 0.0000, 0.0000, 0.0000],
         [0.4704, 0.5296, 0.0000, 0.0000],
         [0.4032, 0.4109, 0.1860, 0.0000],
         [0.2462, 0.2555, 0.2509, 0.2474]]], grad_fn=<SoftmaxBackward0>)


torch.Size([2, 4, 16])